# Data Visualization Workflow
Use this notebook to load an already processed dataset, optionally reload a range file, and open the interactive visualization tools.


In [1]:
# Activate intractive functionality of matplotlib
%matplotlib ipympl
# Activate auto reload 
%load_ext autoreload
%autoreload 2
%reload_ext autoreload
# import libraries
import os
import numpy as np
import subprocess
from ipywidgets import widgets
from IPython.display import display
from ipywidgets import fixed, interact_manual
import warnings
# Ignore all warnings
warnings.filterwarnings("ignore")

# Local module and scripts
from pyccapt.calibration.calibration import widgets as wd
from pyccapt.calibration.data_tools import data_tools
from pyccapt.calibration.tutorials.tutorials_helpers import helper_data_loader
from pyccapt.calibration.tutorials.tutorials_helpers import helper_visualization
from pyccapt.calibration.calibration import share_variables
from pyccapt.calibration.calibration import ion_selection

If HDF5 loading fails because `pytables` is missing, install it in a separate cell before continuing:

`!conda install --yes --prefix {sys.prefix} pytables`


Create the shared `variables` object first. The loaded dataset and range table are stored here for the visualization helpers.


In [2]:
# Create the shared state container used by the visualization helpers.
variables = share_variables.Variables()

Choose the dataset you want to visualize. This notebook works best with data that has already been cropped and calibrated.


In [3]:
button = widgets.Button(description='load dataset')

@button.on_click
def open_file_on_click_r(b):
    global dataset_path
    folder_path = variables.last_directory
    script = '..//..//data_tools//run_dataset_path_qt.py'
    cmd = f"python {script} {folder_path}"
    result = subprocess.run(cmd, capture_output=True, text=True, shell=True)
    dataset_path = result.stdout.strip()
    variables.last_directory = dataset_path

button

Button(description='load dataset', style=ButtonStyle())

## Load Data And Optional Range Files
Set the instrument metadata, load the dataset, and optionally reload a previously saved range table.


Review the dataset settings before loading. The selected values control the derived arrays used by the visualization helpers.


In [5]:
# create an object for selection of instrument specifications of the dataset
tdc, pulse_mode, flight_path_length, t0, max_mc, det_diam = wd.dataset_instrument_specification_selection()

# Display lists and comboboxes to selected instrument specifications
display(tdc, pulse_mode, flight_path_length, t0, max_mc)

Dropdown(description='Data mode:', options=('pyccapt', 'leap_epos', 'leap_pos', 'leap_apt', 'ato_v6'), value='…

Dropdown(description='Pulse mode:', options=('voltage', 'laser'), value='voltage')

FloatText(value=110.0, description='Flight path length:')

FloatText(value=38.0, description='t0:')

FloatText(value=400.0, description='Max mc:')

In [7]:
# Load the dataset with the selected settings and preview the dataset and range table.
helper_data_loader.load_data(dataset_path, max_mc.value, flight_path_length.value, pulse_mode.value, tdc.value, variables, processing_mode=False)
data_tools.extract_data(variables.data, variables, flight_path_length.value, max_mc.value)
display(variables.data)
display(variables.range_data)

The data will be saved on the path: T:/Ortner/03_APT/Oxcart/2454_Apr-07-2025_10-19_NiC2_6082-65min/2454_Apr-07-2025_10-19_NiC2_6082-65min/data_processing/2454_Apr-07-2025_10-19_NiC2_6082-65min/visualization/
The dataset name after saving is: 2454_Apr-07-2025_10-19_NiC2_6082-65min
The figures will be saved on the path: T:/Ortner/03_APT/Oxcart/2454_Apr-07-2025_10-19_NiC2_6082-65min/2454_Apr-07-2025_10-19_NiC2_6082-65min/data_processing/2454_Apr-07-2025_10-19_NiC2_6082-65min/visualization/
Total number of Ions: 22157254
The maximum possible time of flight is: 5010


,x (nm),y (nm),z (nm),mc (Da),mc_uc (Da),high_voltage (V),pulse_v (V),pulse_l (pJ),t (ns),t_c (ns),x_det (cm),y_det (cm),delta_p,multi,start_counter
0,0.0,0.0,0.0,27.981170,31.785536,2232.5865,446.5173,0.0,944.017416,944.017416,2.481633,0.561633,0,1,19612
1,0.0,0.0,0.0,27.434791,31.326274,2232.5865,446.5173,0.0,937.858932,937.858932,2.331429,1.071020,65,1,19677
2,0.0,0.0,0.0,27.408784,31.120324,2232.5865,446.5173,0.0,934.258482,934.258482,2.471837,0.535510,10,1,19687
3,0.0,0.0,0.0,382.462302,445.438904,2231.6625,446.3325,0.0,3334.167576,3334.167576,-0.336327,-0.084898,4294947915,1,306
4,0.0,0.0,0.0,27.372068,31.128182,2231.6625,446.3325,0.0,923.704020,923.704020,1.753469,0.493061,122,1,428
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22157249,0.0,0.0,0.0,27.248027,30.878184,5601.2285,1120.2457,0.0,596.021922,596.021922,-0.055510,1.786122,575,1,8171
22157250,0.0,0.0,0.0,26.959221,29.493772,5601.2285,1120.2457,0.0,580.029066,580.029066,0.894694,-0.907755,3,1,8174
22157251,0.0,0.0,0.0,26.969435,30.591035,5601.2285,1120.2457,0.0,594.019386,594.019386,-0.120816,1.854694,10,1,8184
22157252,0.0,0.0,0.0,26.999729,29.604551,5601.2285,1120.2457,0.0,580.015350,580.015350,0.790204,-0.731429,11,1,8195


,name,ion,mass,mc,mc_low,mc_up,color,element,complex,isotope,charge
0,unranged0,un,0.0,0.0,0.0,400.0,#000000,[unranged],[0],[0],0


If you already have a saved range file, load it here so the ion labels and colors are restored before visualization.


In [7]:
button_r = widgets.Button(description='load range dataset')

@button_r.on_click
def open_file_on_click_r(b):
    global range_path
    folder_path = variables.last_directory
    script = '..//..//data_tools//run_dataset_path_qt.py'
    cmd = f"python {script} {folder_path}"
    result = subprocess.run(cmd, capture_output=True, text=True, shell=True)
    range_path = result.stdout.strip()

button_r

Button(description='load range dataset', style=ButtonStyle())

In [8]:
# If a range file was chosen, load it into the shared state and preview it.
if 'range_path' in globals():
    variables.range_data = data_tools.read_range(range_path)

In [9]:
# Preview the range table with colors applied before saving or visualization.
display(variables.range_data.style.applymap(ion_selection.display_color, subset=['color']))

,ion,mass,mc,mc_low,mc_up,color,element,complex,isotope,charge
0,${}^{1}H^{+}$,1.010000,1.000980,0.945341,1.166101,#b2aa2d,['H'],[1],[1],1
1,${}^{2}H^{+}$,2.010000,2.001920,1.946281,2.142513,#b2aa2d,['H'],[1],[2],1
2,${}^{12}C^{2+}$,6.000000,5.905587,5.837683,6.144295,#daa460,['C'],[1],[12],2
3,${}^{12}C^{+}$,12.000000,11.911227,11.855588,12.174464,#daa460,['C'],[1],[12],1
4,${}^{13}C^{+}$,13.000000,12.912167,12.881057,13.175404,#daa460,['C'],[1],[13],1
5,${}^{27}Al^{2+}$,13.490000,13.412637,13.356999,13.663610,#e7e0d1,['Al'],[1],[27],2
6,${}^{28}Si^{2+}$,13.990000,13.913107,13.869733,14.164080,#429806,['Si'],[1],[28],2
7,${}^{29}Si^{2+}$,14.490000,14.413578,14.357939,14.676815,#429806,['Si'],[1],[29],2
8,${}^{30}Si^{2+}$,14.985000,14.914048,14.882938,15.177285,#429806,['Si'],[1],[30],2
9,${}^{17}O^{+}$,17.000000,17.002406,16.922239,17.093941,#1f963a,['O'],[1],[17],1


Save the current range table from the widget below once the ion list looks correct.


In [ ]:
# Save the current range table interactively.
interact_manual_range = interact_manual.options(manual_name="save range")
interact_manual_range(data_tools.save_range, variables=fixed(variables));

In [10]:
variables.data

,x (nm),y (nm),z (nm),mc (Da),mc_uc (Da),high_voltage (V),pulse_v (V),pulse_l (pJ),t (ns),t_c (ns),x_det (cm),y_det (cm),delta_p,multi,start_counter
0,0.0,0.0,0.0,27.981170,31.785536,2232.5865,446.5173,0.0,944.017416,944.017416,2.481633,0.561633,0,1,19612
1,0.0,0.0,0.0,27.434791,31.326274,2232.5865,446.5173,0.0,937.858932,937.858932,2.331429,1.071020,65,1,19677
2,0.0,0.0,0.0,27.408784,31.120324,2232.5865,446.5173,0.0,934.258482,934.258482,2.471837,0.535510,10,1,19687
3,0.0,0.0,0.0,382.462302,445.438904,2231.6625,446.3325,0.0,3334.167576,3334.167576,-0.336327,-0.084898,4294947915,1,306
4,0.0,0.0,0.0,27.372068,31.128182,2231.6625,446.3325,0.0,923.704020,923.704020,1.753469,0.493061,122,1,428
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22157249,0.0,0.0,0.0,27.248027,30.878184,5601.2285,1120.2457,0.0,596.021922,596.021922,-0.055510,1.786122,575,1,8171
22157250,0.0,0.0,0.0,26.959221,29.493772,5601.2285,1120.2457,0.0,580.029066,580.029066,0.894694,-0.907755,3,1,8174
22157251,0.0,0.0,0.0,26.969435,30.591035,5601.2285,1120.2457,0.0,594.019386,594.019386,-0.120816,1.854694,10,1,8184
22157252,0.0,0.0,0.0,26.999729,29.604551,5601.2285,1120.2457,0.0,580.015350,580.015350,0.790204,-0.731429,11,1,8195


Use the next widget to export the current dataset in any additional formats you need.


In [32]:
# Save the dataset interactively in the output formats you want.
interact_manual_data = interact_manual.options(manual_name="save data")
interact_manual_data(data_tools.save_data, data=fixed(variables.data), variables=fixed(variables),
                name=widgets.Text(value=variables.result_data_name),
                hdf=widgets.Dropdown(options=[('True', True), ('False', False)]),
                epos=widgets.Dropdown(options=[('False', False), ('True', True)]),
                pos=widgets.Dropdown(options=[('False', False), ('True', True)]),
                ato_6v=widgets.Dropdown(options=[('False', False), ('True', True)]),
                csv=widgets.Dropdown(options=[('False', False), ('True', True)]),
               temp=fixed(False));

interactive(children=(Text(value='R5096_92775_C4', continuous_update=False, description='name'), Dropdown(desc…

## Visualization
The final cells refresh the extracted arrays and open the main interactive visualization interface.


In [34]:
# Refresh the extracted arrays before launching the visualization helper.
data_tools.extract_data(variables.data, variables, flight_path_length.value, max_mc.value)

The maximum possible time of flight is: 5010


In [26]:
# Open the main visualization interface.
helper_visualization.call_visualization(variables)

In [ ]:
from pyccapt.calibration.calibration import mc_plot
hist = variables.data['mc (Da)'].to_numpy()
mc_hist = mc_plot.AptHistPlotter(hist[hist < 40], variables)
mc_hist.plot_histogram(bin_width=0.1, label='mc', steps='stepfilled', log=True)
mc_hist.find_peaks_and_widths()
mc_hist.plot_hist_info_legend(loc='right')
mc_hist.plot_line_hist()
mc_hist.manual_background_fit()

In [ ]:
# Display the plotly reconstruction object so you can inspect or embed it.
variables.plotly_3d_reconstruction

In [ ]:
from IPython.display import display, HTML
display(HTML(variables.animation_detector_html))